# Vision pipelines

[Back to the introduction](ch01_demo.ipynb). Run in order. `LLM.pipeline(task, model=...)` also loads Transformers.js vision tasks: image classification, zero-shot image classification, image features and object detection. Each section loads a small browser model, runs it on two images, and releases it before the next one; the last section runs object detection on random photos and draws the boxes with Pillow. Image tasks accept an HTTP(S) URL, encoded JPEG/PNG/WebP bytes, a file in the JupyterLite file browser, a `uint8` NumPy array or a PIL image. Encoded bytes are the cheapest to send to the browser. The native backend does not run vision tasks.

In [ ]:
from IPython.display import Image, display

from lmlite import LLM

base = "https://huggingface.co/datasets/Xenova/transformers.js-docs/resolve/main"
images = {"tiger": f"{base}/tiger.jpg", "cats": f"{base}/cats.jpg"}
for url in images.values():
    display(Image(url=url, width=240))

## Image classification

A ViT model scores the 1,000 ImageNet classes. `top_k` limits the labels returned.

In [ ]:
classifier = await LLM.pipeline("image-classification", model="vit-base-patch16-224", backend="wasm")
for name, url in images.items():
    print(name, [(item["label"], round(item["score"], 3)) for item in await classifier(url, top_k=3)])
await classifier.dispose()

## Zero-shot image classification

CLIP ranks labels you choose at call time. Phrases such as "a photo of ..." work better than single words.

In [ ]:
clip = await LLM.pipeline("zero-shot-image-classification", model="clip-vit-base-patch32", backend="wasm")
labels = ["a photo of a tiger", "a photo of cats", "a photo of food"]
for name, url in images.items():
    result = await clip(url, candidate_labels=labels)
    print(name, [(item["label"], round(item["score"], 3)) for item in result])
await clip.dispose()

## Image features

DINOv2 turns each image into one vector (the first, CLS, token). Pass a list to embed several images in one call, then compare them with cosine similarity.

In [ ]:
import math

extractor = await LLM.pipeline("image-feature-extraction", model="dinov2-small", backend="wasm")
tiger, cats = await extractor(list(images.values()))
dot = sum(a * b for a, b in zip(tiger, cats))
print(len(tiger), "dimensions; cosine similarity:", round(dot / math.hypot(*tiger) / math.hypot(*cats), 3))
await extractor.dispose()

## Object detection

DETR finds the 80 kinds of everyday objects of the COCO dataset (people, cats, dogs, bicycles, buses, chairs, bottles, and so on) and returns each box in pixels (`xmin`, `ymin`, `xmax`, `ymax`), with a label and a confidence score. Raise `threshold` to keep only confident detections. It cannot name anything outside those 80 classes, so some photos show nothing.

The next cells draw the boxes with Pillow, on random photos from Wikimedia Commons (each with its own licence, printed with the author). Pillow comes with `pip install 'lmlite[images]'`; the JupyterLite sites include it.

In [ ]:
import io
import json
import random
import re
import urllib.parse
import urllib.request
from collections import Counter

from PIL import Image as PilImage
from PIL import ImageDraw, ImageFont, ImageOps

try:
    import pyodide_http

    pyodide_http.patch_all()  # urllib goes through the browser's fetch
except ImportError:
    pass

CATEGORIES = [
    "Featured pictures of people",
    "Featured pictures of sports",
    "Featured pictures of cats",
    "Featured pictures of dogs",
    "Featured pictures of bicycles",
    "Featured pictures of buses",
]


def fetch(url):
    request = urllib.request.Request(url, headers={"User-Agent": "lmlite-notebook"})
    with urllib.request.urlopen(request) as response:
        return response.read()


def commons(**parameters):
    query = urllib.parse.urlencode({"action": "query", "format": "json", "origin": "*", **parameters})
    return json.loads(fetch(f"https://commons.wikimedia.org/w/api.php?{query}"))["query"]


titles = []
for category in CATEGORIES:
    members = commons(list="categorymembers", cmtitle=f"Category:{category}", cmtype="file", cmlimit=500)
    titles += [member["title"] for member in members["categorymembers"]]
titles = list(dict.fromkeys(titles))  # a photo can be in several categories
print(len(titles), "photos to pick from")

# The photos shown so far, by title: (details, downloaded bytes). The next pick never repeats them.
seen = {}

In [ ]:
detector = await LLM.pipeline("object-detection", model="detr-resnet-50", backend="wasm")

## Draw the detections

`annotate` draws a box per detection, in one colour per label, with a strip above it reading for example `cat 0.99`. `contact_sheet` lays photos out in a grid, numbered, so the photos and their annotated copies line up.

In [ ]:
# The built-in bitmap font is tiny but works everywhere: loading a FreeType font hangs in xeus-python.
FONT = ImageFont.load_default_imagefont()
COLORS = ["#e6194b", "#3cb44b", "#4363d8", "#f58231", "#911eb4", "#008080", "#9a6324", "#800000"]


def badge(text, color, scale):
    """White text on a coloured strip, enlarged `scale` times."""
    left, top, right, bottom = FONT.getbbox(text)
    mask = PilImage.new("L", (right, bottom), 0)
    ImageDraw.Draw(mask).text((0, 0), text, font=FONT, fill=255)
    mask = mask.resize((mask.width * scale, mask.height * scale), PilImage.NEAREST)
    strip = PilImage.new("RGB", (mask.width + 8, mask.height + 8), color)
    strip.paste("white", (4, 4), mask)
    return strip


def annotate(image, detections):
    annotated = image.copy()
    draw = ImageDraw.Draw(annotated)
    colors = {}
    for item in detections:
        box = item["box"]
        color = colors.setdefault(item["label"], COLORS[len(colors) % len(COLORS)])
        draw.rectangle((box["xmin"], box["ymin"], box["xmax"], box["ymax"]), outline=color, width=4)
        strip = badge(f"{item['label']} {item['score']:.2f}", color, scale=2)
        annotated.paste(strip, (round(box["xmin"]), max(0, round(box["ymin"]) - strip.height)))  # above the box
    return annotated


def contact_sheet(images, columns=4, tile=340):
    """The images in a grid of square tiles, each numbered from 1."""
    rows = -(-len(images) // columns)
    sheet = PilImage.new("RGB", (columns * (tile + 6) + 6, rows * (tile + 6) + 6), "white")
    for number, image in enumerate(images):
        fitted = ImageOps.contain(image, (tile, tile))
        left, top = 6 + (number % columns) * (tile + 6), 6 + (number // columns) * (tile + 6)
        sheet.paste(fitted, (left + (tile - fitted.width) // 2, top + (tile - fitted.height) // 2))
        sheet.paste(badge(str(number + 1), "black", scale=3), (left, top))
    return sheet

## Eight random photos

Run the next cell as often as you like: each run picks 8 photos you have not seen yet, shows them as one numbered block, runs the detector on all of them with `map()` (4 per batch, reporting progress) and shows the same 8 photos again with their boxes, labels and confidence scores. A photo with nothing found says so. `on_error="return"` means one photo that fails does not lose the others.

The photos shown so far stay in `seen` (title to details and downloaded bytes), so no photo comes back until all of them have been shown, and then the notebook starts over. `seen` lives as long as the kernel.

In [ ]:
from IPython.display import display

count = 8
unseen = [title for title in titles if title not in seen]
if len(unseen) < count:
    print(f"All {len(titles)} photos have been shown: starting over")
    seen.clear()
    unseen = titles
pages = commons(
    titles="|".join(random.sample(unseen, count)),
    prop="imageinfo",
    iiprop="url|extmetadata",
    iiurlwidth=480,
    iiextmetadatafilter="LicenseShortName|Artist",
)["pages"].values()
photos = [(page["title"], page["imageinfo"][0]) for page in pages if "imageinfo" in page]
files = [fetch(info["thumburl"]) for _, info in photos]
seen.update({title: (info, file) for (title, info), file in zip(photos, files)})
print(f"{len(seen)} of {len(titles)} photos seen")
images = [PilImage.open(io.BytesIO(file)).convert("RGB") for file in files]

print("The photos:")
display(contact_sheet(images))

results = await detector.map(
    files,
    batch_size=4,
    threshold=0.7,
    on_error="return",
    progress_callback=lambda done, total: print(f"detected {done}/{total}"),
)

print("With detections:")
display(contact_sheet([annotate(image, [] if isinstance(r, Exception) else r) for image, r in zip(images, results)]))

for number, ((title, info), detections) in enumerate(zip(photos, results), 1):
    meta = info["extmetadata"]
    author = re.sub("<[^>]+>", "", meta.get("Artist", {}).get("value", "unknown author"))
    licence = meta.get("LicenseShortName", {}).get("value", "see the file page")
    if isinstance(detections, Exception):
        outcome = f"detection failed ({detections})"
    elif not detections:
        outcome = "nothing found"
    else:
        counts = Counter(d["label"] for d in detections)
        outcome = f"{len(detections)} object(s): " + ", ".join(f"{label} x{n}" if n > 1 else label for label, n in counts.items())
    print(f"{number}. {title}: {outcome} (photo: {author}, {licence}, Wikimedia Commons)")

In [ ]:
await detector.dispose()

The inference pipelines are released; cached files are retained for subsequent runs. See the usage guide for task options and output formats.